In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Playground S6E7 — Student Health Risk (FE + SHAP selection + FT-Transformer stack)

A stacked ensemble for the 3-class, balanced-accuracy **student health risk** problem
(`unhealthy` / `at-risk` / `fit`), heavily imbalanced toward `at-risk`.

**Pipeline:**
1. **Feature engineering** — ratio/interaction features, nonlinear transforms, and a large
   family of *categorical variables derived from the numeric columns* (domain-threshold bins
   **and** train-fitted quantile bins), motivated directly by the companion EDA notebook.
2. **SHAP-based feature selection** — a quick XGBoost, CatBoost, and LightGBM are each fit on the
   target-encoded feature matrix, TreeExplainer SHAP values are computed per model, normalized, and
   averaged; per-encoded-column importances are folded back onto their *source* feature, and the
   smallest feature set reaching 95% cumulative averaged |SHAP| is kept.
3. **Five base learners** → multinomial-LR meta-learner:
   **XGBoost + CatBoost + Logistic Regression + FT-Transformer + LightGBM.**
4. **Seed-bagged outer stack** — the whole 5-fold stacking pass runs once per seed in `SEEDS`
   (fold splits *and* every learner's own randomness vary), and OOF/test predictions are averaged
   across seeds before the meta-learner, trading GPU time for lower variance.
5. **Cross-fitted isotonic calibration** of each base learner's OOF probabilities, kept only if it
   improves meta-learner OOF balanced accuracy.
6. **Per-class decision-weight search** on the meta-learner's OOF probabilities — a coordinate-ascent
   grid search that generalizes the old binary prior-correction toggle.

**Base learners and why they're here.** The tree models (XGB, CatBoost, LightGBM) do the heavy
lifting via three different split-finding strategies and two different categorical-handling
mechanisms (target encoding vs. native); LR is a fast linear baseline; the **FT-Transformer**
(Gorishniy et al., 2021, *Revisiting Deep Learning Models for Tabular Data*) is a neural learner
included mainly for **diversity** — it makes different errors than the trees, which is what helps a
stack even when a learner's standalone score is similar. It's a **from-scratch PyTorch
implementation** (no third-party tabular-DL packages), so it needs no extra pip installs beyond what
Kaggle ships with.

> **Note on the previous version.** The earlier RealMLP + TabTransformer learners have been removed.
> `pytabkit` (RealMLP) won't install offline on Kaggle, and rather than add another dependency we
> replace both neural learners with the self-contained FT-Transformer below, plus LightGBM as a
> cheap, mechanism-diverse tree learner.

## What carries over unchanged
Multiclass OOF target encoding, HPO on a stratified sample, balanced-accuracy optimization,
per-fold leakage discipline, and GPU auto-detection all carry over.

## EDA facts this notebook relies on (from `s6e7-eda-baseline`)
- **Heavy target imbalance** — predicting `at-risk` for everything scores ~86% raw accuracy but
  only ~33% *balanced* accuracy, so every learner is class-balanced (weights).
- **Missingness is essentially random (MCAR)** — so imputation carries no signal; trees see raw
  NaNs natively, and the linear/neural branches median-impute only to satisfy their input contract.
- **No train/test distribution shift** — numeric and categorical distributions overlay almost
  perfectly, so transforms fitted on train (quantile edges, scalers, vocabularies) apply safely to test.
- **Most class-separable numerics:** `sleep_duration`, `bmi`, `exercise_duration`, `step_count` —
  these get the most derived features below.

## GPU support
XGBoost, CatBoost, and the FT-Transformer use the GPU when one is available (**Settings →
Accelerator → GPU**). LightGBM runs on CPU deliberately — Kaggle's stock wheel isn't built with GPU
support, and CPU LightGBM is fast enough not to matter. Detection is via `torch.cuda.is_available()`;
with no GPU everything falls back to CPU, with the FT-Transformer benefiting most from the GPU.


In [ ]:
!pip install -q optuna catboost shap

In [ ]:
import os, time, warnings, json
_NOTEBOOK_T0 = time.time()  # brackets true pipeline wall-time for the run-metrics summary at the end
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.utils import resample
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
import torch
import torch.nn as nn
import optuna, shap

warnings.filterwarnings("ignore"); optuna.logging.set_verbosity(optuna.logging.WARNING)
RNG, N_FOLDS, N_TRIALS = 42, 5, 25
SEEDS = [42]                      # outer-stack seed bagging (single seed this run to cut runtime); RNG stays fixed for HPO/feature selection
TARGET = "health_condition"

# --- GPU configuration -------------------------------------------------------
# Set the Kaggle notebook Accelerator to "GPU" (Settings > Accelerator) to use
# this. Detected once via torch; each library gets its GPU flags below.
USE_GPU        = torch.cuda.is_available()
TORCH_DEVICE   = "cuda" if USE_GPU else "cpu"
print(f"GPU available: {USE_GPU}"
      + (f"  ({torch.cuda.get_device_name(0)})" if USE_GPU else ""))

def xgb_device_kwargs():
    # XGBoost >= 2.0: device='cuda' + tree_method='hist' (gpu_hist is deprecated)
    return {"tree_method": "hist", "device": "cuda" if USE_GPU else "cpu"}

def cat_device_kwargs():
    # CatBoost: task_type='GPU', devices='0' when a GPU is available
    return {"task_type": "GPU", "devices": "0"} if USE_GPU else {"task_type": "CPU"}

def lgb_device_kwargs():
    # LightGBM: deliberately CPU-only. Kaggle's stock wheel isn't built with GPU
    # support, and requesting device_type="gpu" against a CPU-only build fails
    # hard rather than falling back -- CPU LightGBM is fast enough not to matter.
    return {"device_type": "cpu"}


## 1. Load data and encode the target

Column-type detection is deferred until **after** feature engineering, so the new derived categoricals are routed correctly.

In [ ]:
df      = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/train.csv")
df_test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/test.csv")
print("Train:", df.shape, " Test:", df_test.shape)

classes    = sorted(df[TARGET].unique())
cls_to_int = {c: i for i, c in enumerate(classes)}
int_to_cls = {i: c for c, i in cls_to_int.items()}
N_CLASSES  = len(classes)
print(f"Classes ({N_CLASSES}): {classes}")

y        = df[TARGET].map(cls_to_int).values
X        = df.drop(columns=["id", TARGET]).reset_index(drop=True)
X_test   = df_test.drop(columns=["id"]).reset_index(drop=True)
test_ids = df_test["id"].values

# raw schema (used by the feature-engineering block below)
RAW_NUM = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
           "step_count", "exercise_duration", "water_intake"]
RAW_CAT = ["diet_type", "stress_level", "sleep_quality",
           "physical_activity_level", "smoking_alcohol", "gender"]
RAW_NUM = [c for c in RAW_NUM if c in X.columns]
RAW_CAT = [c for c in RAW_CAT if c in X.columns]
print("Raw numeric:", RAW_NUM)
print("Raw categorical:", RAW_CAT)
print("Class balance:")
for c in classes: print(f"  {c}: {(df[TARGET] == c).mean():.3f}")

## 2. Feature engineering

Everything here is **unsupervised** (no target used), so it's leakage-safe to compute on train and
apply identically to test — and because the EDA showed *no train/test distribution shift*, quantile
edges fitted on train transfer cleanly to test.

Three families of features, plus a deliberately large set of derived categoricals so that the SHAP
selection step below has something meaningful to prune.

**(a) Ratio & interaction features (numeric).** The raw columns are individually informative, but
*combinations* often carry the physiological signal — e.g. calories burned *per minute of exercise*
(intensity), or heart rate *relative to* step count (a sedentary-but-elevated-HR signature). These
give the trees ready-made interactions they'd otherwise have to discover through deep splits:
`exercise_intensity`, `hydration_ratio`, `activity_ratio`, `cal_per_step`, `cal_per_kg_bmi`,
`steps_per_hr`, `water_per_cal`, `ex_per_step`, `metabolic_load`, `sedentary_index`,
`cardio_stress`, `activity_balance`, plus two composite scores (`fitness_score`, `sleep_hr_product`,
`ex_water_product`).

**(b) Nonlinear transforms (numeric).** `bmi_sq` captures the U-shaped health risk at both BMI
extremes; `log_steps` / `log_cal` tame right-skew; `sleep_dev` and `hr_dev` encode *distance from a
healthy set-point* (8 h sleep, ~70 bpm resting HR) rather than the raw level — a monotone feature is
easier for the linear/neural learners than a bump-shaped one.

**(c) Categorical variables *from* numeric columns (the requested transformation).** Two ways:
- **Domain-threshold bins** using clinical cut-points: `bmi_category` (under/normal/over/obese),
  `sleep_category`, `heart_rate_category`, `step_category`, `water_category`, `exercise_category`,
  `calorie_category`. These let the tree/neural learners treat "clinically overweight" as a discrete
  state, and — crucially — they feed the **multiclass target encoder**, which turns each bin into
  per-class rates (a smooth, powerful signal the raw float can't express directly).
- **Quantile bins** (`*_qbin`, quintiles) for the most separable numerics identified in the EDA
  (`sleep_duration`, `bmi`, `exercise_duration`, `step_count`, …). Edges are fit on **train only**
  and reused on test, so they're leakage-safe.

Binning the same numeric several different ways is intentionally redundant — the SHAP step keeps the
cuts that matter and drops the rest.


In [ ]:
EPS = 1e-3
def _safe(a, b): return a / (b + EPS)

def engineer_features(d):
    """Unsupervised feature engineering. Safe to apply independently to train and test."""
    d = d.copy()
    sleep, hr, bmi = d["sleep_duration"], d["heart_rate"], d["bmi"]
    cal,   steps   = d["calorie_expenditure"], d["step_count"]
    ex,    water   = d["exercise_duration"], d["water_intake"]

    # (a) ratios / interactions
    d["exercise_intensity"] = _safe(cal, ex)        # kcal per exercise-minute
    d["hydration_ratio"]    = _safe(water, steps)   # water per step
    d["activity_ratio"]     = _safe(ex, sleep)      # exercise vs sleep
    d["cal_per_step"]       = _safe(cal, steps)
    d["cal_per_kg_bmi"]     = _safe(cal, bmi)
    d["steps_per_hr"]       = _safe(steps, hr)
    d["water_per_cal"]      = _safe(water, cal)
    d["ex_per_step"]        = _safe(ex, steps)
    d["sleep_hr_product"]   = sleep * hr
    d["ex_water_product"]   = ex * water
    d["metabolic_load"]     = _safe(cal, sleep)     # burn per sleep-hour
    d["sedentary_index"]    = _safe(hr, steps)      # elevated HR, low movement
    d["cardio_stress"]      = _safe(hr, sleep)
    d["activity_balance"]   = _safe(ex, ex + sleep)
    d["fitness_score"]      = steps / 1000.0 + ex - _safe(bmi, 1.0) + sleep

    # (b) nonlinear transforms
    d["bmi_sq"]    = bmi ** 2
    d["log_steps"] = np.log1p(steps.clip(lower=0))
    d["log_cal"]   = np.log1p(cal.clip(lower=0))
    d["sleep_dev"] = (sleep - 8.0).abs()            # distance from ideal 8h
    d["hr_dev"]    = (hr - 70.0).abs()              # distance from ~70 bpm resting

    # (c1) domain-threshold categoricals from numeric (clinical cut-points)
    d["bmi_category"] = pd.cut(bmi, [-np.inf, 18.5, 25, 30, np.inf],
                               labels=["underweight", "normal", "overweight", "obese"]).astype("object")
    d["sleep_category"] = pd.cut(sleep, [-np.inf, 5, 8, np.inf],
                                 labels=["short", "normal", "long"]).astype("object")
    d["heart_rate_category"] = pd.cut(hr, [-np.inf, 60, 100, np.inf],
                                      labels=["low", "normal", "high"]).astype("object")
    d["step_category"] = pd.cut(steps, [-np.inf, 5000, 10000, np.inf],
                                labels=["low", "medium", "high"]).astype("object")
    d["water_category"] = pd.cut(water, [-np.inf, 1.5, 3, np.inf],
                                 labels=["low", "medium", "high"]).astype("object")
    d["exercise_category"] = pd.cut(ex, [-np.inf, 15, 45, np.inf],
                                    labels=["light", "moderate", "intense"]).astype("object")
    d["calorie_category"] = pd.cut(cal, [-np.inf, 1800, 2500, np.inf],
                                   labels=["low", "medium", "high"]).astype("object")
    # (d) targeted additions (run-log SHAP showed signal concentrates in the strong
    #     raw features; these combine/transform those rather than add weak ratios)
    #  d1) deviation / interaction transforms of the strongest numerics
    d["bmi_dev"]          = (bmi - 22.0).abs()          # monotone distance from healthy-center BMI
    d["sleep_dev_sq"]     = d["sleep_dev"] ** 2         # sharpen the sleep U-shape
    d["sleep_bmi_dev"]    = d["sleep_dev"] * d["bmi_dev"]   # compound distance-from-healthy
    d["health_dev_score"] = d["sleep_dev"] + d["bmi_dev"] + d["hr_dev"]  # additive set-point risk
    #  d2) categorical crosses of the high-|SHAP| categoricals (target-encoded downstream)
    d["stress_x_sleepqual"] = d["stress_level"].astype(str) + "_" + d["sleep_quality"].astype(str)
    d["stress_x_activity"]  = d["stress_level"].astype(str) + "_" + d["physical_activity_level"].astype(str)
    d["activity_x_smoking"] = d["physical_activity_level"].astype(str) + "_" + d["smoking_alcohol"].astype(str)
    d["diet_x_activity"]    = d["diet_type"].astype(str) + "_" + d["physical_activity_level"].astype(str)
    d["stress_x_smoking"]   = d["stress_level"].astype(str) + "_" + d["smoking_alcohol"].astype(str)
    return d

# (c2) quantile-bin categoricals — edges fit on TRAIN only, reused on test (leakage-safe)
QBIN_COLS = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
             "step_count", "exercise_duration", "water_intake",
             "fitness_score", "metabolic_load", "exercise_intensity",
             "health_dev_score"]

def fit_quantile_edges(d, cols, q=5):
    edges = {}
    for c in cols:
        e = np.unique(np.nanquantile(d[c].astype(float), np.linspace(0, 1, q + 1)))
        e[0], e[-1] = -np.inf, np.inf            # open the tails for unseen test values
        edges[c] = e
    return edges

def apply_quantile_bins(d, edges):
    d = d.copy()
    for c, e in edges.items():
        labs = [f"q{i}" for i in range(len(e) - 1)]
        d[f"{c}_qbin"] = pd.cut(d[c].astype(float), bins=e, labels=labs,
                                include_lowest=True).astype("object")
    return d

# apply FE
X      = engineer_features(X)
X_test = engineer_features(X_test)
qedges = fit_quantile_edges(X, QBIN_COLS, q=5)     # train-only edges
X      = apply_quantile_bins(X, qedges)
X_test = apply_quantile_bins(X_test, qedges)
print(f"After FE:  train {X.shape},  test {X_test.shape}")
print(f"Added {X.shape[1] - len(RAW_NUM) - len(RAW_CAT)} engineered columns")

### 2.1 Detect column types on the engineered frame

Same auto-detector as before, run *after* FE so the derived bins are treated as categorical (and later target-encoded) while ratios/transforms stay numeric. Categorical NaNs become an explicit `"nan"` string via the cast — missingness is a level, not a hole.

In [ ]:
def _is_stringy(s):
    if isinstance(s.dtype, pd.CategoricalDtype):   return True
    if s.dtype == object:                          return True
    if s.dtype.kind in ("O", "U", "S", "T"):       return True
    return pd.api.types.is_string_dtype(s) and not pd.api.types.is_numeric_dtype(s)

def detect_column_types(frame):
    cat = [c for c in frame.columns
           if _is_stringy(frame[c]) or (frame[c].dtype.kind in "iu" and frame[c].nunique() <= 12)]
    num = [c for c in frame.columns if c not in cat]
    return cat, num

CAT_COLS, NUM_COLS = detect_column_types(X)
for c in CAT_COLS:
    X[c] = X[c].astype(str); X_test[c] = X_test[c].astype(str)

print(f"Categorical ({len(CAT_COLS)}): {CAT_COLS}")
print(f"Numeric ({len(NUM_COLS)}): {NUM_COLS}")

## 3. Multiclass OOF target encoding + balanced sample weights

Each categorical column expands into K encoded columns (per-class rate), OOF + smoothed to prevent leakage. `balanced_sample_weight` gives the tree/linear learners class balancing; RealMLP gets balancing via resampling instead (below).

In [ ]:
def oof_target_encode_multiclass(X_tr, y_tr, X_te, cat_cols, n_classes,
                                 n_splits=5, seed=RNG, m=20.0):
    X_tr_enc = X_tr.drop(columns=cat_cols).copy()
    X_te_enc = X_te.drop(columns=cat_cols).copy()
    priors = np.array([(y_tr == k).mean() for k in range(n_classes)])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for col in cat_cols:
        oof = np.tile(priors, (len(X_tr), 1)).astype(float)
        for tr_idx, val_idx in skf.split(X_tr, y_tr):
            sub = pd.DataFrame({col: X_tr[col].iloc[tr_idx].values})
            for k in range(n_classes):
                sub[f"y{k}"] = (y_tr[tr_idx] == k).astype(float)
            agg = sub.groupby(col).agg(["sum", "count"])
            for k in range(n_classes):
                s = agg[(f"y{k}", "sum")]; c = agg[(f"y{k}", "count")]
                enc = (s + priors[k] * m) / (c + m)
                oof[val_idx, k] = X_tr[col].iloc[val_idx].map(enc).fillna(priors[k]).values
        for k in range(n_classes):
            X_tr_enc[f"{col}__te{k}"] = oof[:, k]

        sub = pd.DataFrame({col: X_tr[col].values})
        for k in range(n_classes):
            sub[f"y{k}"] = (y_tr == k).astype(float)
        agg = sub.groupby(col).agg(["sum", "count"])
        for k in range(n_classes):
            s = agg[(f"y{k}", "sum")]; c = agg[(f"y{k}", "count")]
            enc = (s + priors[k] * m) / (c + m)
            X_te_enc[f"{col}__te{k}"] = X_te[col].map(enc).fillna(priors[k]).values
    return X_tr_enc, X_te_enc

def balanced_sample_weight(y_arr):
    return compute_sample_weight(class_weight="balanced", y=y_arr)

## 4. SHAP-based feature selection — averaged across XGBoost, CatBoost, and LightGBM

Feature engineering above produced a lot of columns, many deliberately redundant. Rather than guess
which survive, we let three different tree models *tell us* via SHAP, and average their opinions
instead of trusting one model's view of what matters.

**Method.**
1. Take a stratified 75/25 split; **multiclass-target-encode** the categoricals (5-fold OOF on the
   75% slice, applied to the 25% slice) so every column is numeric and leakage-safe.
2. Fit three quick, class-balanced models on the same encoded matrix: XGBoost, CatBoost, LightGBM.
3. Compute **TreeExplainer SHAP values** on the held-out 25% for each model separately. For a 3-class
   model SHAP returns one attribution per class; we take **mean |SHAP| across both samples and
   classes** as a single importance per *encoded* column, per model.
4. Each categorical expands into `K` target-encoded columns (`col__te0..te{K-1}`); we **sum their
   importances back onto the source feature**, so a categorical is judged as a whole rather than
   penalized for being split across columns.
5. **Normalize each model's importance ranking to sum to 1**, then average the three — this prevents
   whichever model happens to produce larger raw SHAP magnitudes from dominating. Each model's own
   top-20 is printed separately below for observability, alongside the averaged ranking actually used.
6. Rank original features by the averaged importance and keep the smallest set reaching
   **`SHAP_CUM` (95%)** of total importance, with a floor of `MIN_FEATURES`.

**Why average three models.** A single model's SHAP ranking reflects that model's particular biases
(e.g. XGBoost's depth-first greedy splits vs. LightGBM's leaf-wise growth vs. CatBoost's ordered
boosting) — features useful to one tree family but not another were previously at risk of being
dropped for the whole pipeline, including for LR and the FT-Transformer, which have no say in a
single-model selection. Averaging three different tree families gives a more robust consensus.

> Note: SHAP importance measures *contribution to these particular models*, not ground-truth causal
> relevance. It's a pragmatic filter, not a causal claim.


In [ ]:
SHAP_CUM     = 0.95     # keep features up to this cumulative share of |SHAP|
MIN_FEATURES = 12       # never drop below this many original features

# 1) stratified split + OOF target encoding (leakage-safe)
X_sel_tr, X_sel_va, y_sel_tr, y_sel_va = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RNG)
X_sel_tr = X_sel_tr.reset_index(drop=True); X_sel_va = X_sel_va.reset_index(drop=True)
Xtr_e, Xva_e = oof_target_encode_multiclass(
    X_sel_tr, y_sel_tr, X_sel_va, CAT_COLS, N_CLASSES, n_splits=5)

# 2) three fast, comparably-sized, class-balanced models on the same encoded matrix
sel_xgb = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
    colsample_bytree=0.8, objective="multi:softprob", num_class=N_CLASSES,
    eval_metric="mlogloss", **xgb_device_kwargs(), random_state=RNG,
    n_jobs=-1, verbosity=0)
sel_xgb.fit(Xtr_e, y_sel_tr, sample_weight=balanced_sample_weight(y_sel_tr))

sel_cat = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6, loss_function="MultiClass",
    auto_class_weights="Balanced", random_seed=RNG, **cat_device_kwargs(),
    verbose=0, allow_writing_files=False)
sel_cat.fit(Xtr_e, y_sel_tr)

sel_lgb = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
    subsample_freq=1, colsample_bytree=0.8, objective="multiclass",
    num_class=N_CLASSES, class_weight="balanced", random_state=RNG,
    **lgb_device_kwargs(), n_jobs=-1, verbosity=-1)
sel_lgb.fit(Xtr_e, y_sel_tr)

SEL_MODELS = {"xgb": sel_xgb, "cat": sel_cat, "lgb": sel_lgb}

# 3) TreeExplainer SHAP per model -> mean|SHAP| per encoded column
def _encoded_importance(model, Xva_e):
    sv = np.asarray(shap.TreeExplainer(model).shap_values(Xva_e))
    if sv.ndim == 3 and sv.shape[0] == N_CLASSES:      # (K, n, f)
        return np.abs(sv).mean(axis=(0, 1))
    elif sv.ndim == 3 and sv.shape[-1] == N_CLASSES:   # (n, f, K)
        return np.abs(sv).mean(axis=(0, 2))
    else:                                              # (n, f)
        return np.abs(sv).mean(axis=0)

# 4) fold target-encoded columns back onto their source categorical
import re
def _to_original(colname):
    m = re.match(r"^(.*)__te\d+$", colname)
    return m.group(1) if m else colname

enc_cols = list(Xva_e.columns)
per_model_imp = {}   # name -> Series of original-feature importance, normalized to sum 1
for name, model in SEL_MODELS.items():
    enc_imp = _encoded_importance(model, Xva_e)
    assert len(enc_imp) == len(enc_cols)
    orig_imp = {}
    for col, v in zip(enc_cols, enc_imp):
        o = _to_original(col)
        orig_imp[o] = orig_imp.get(o, 0.0) + float(v)
    ser = pd.Series(orig_imp).sort_values(ascending=False)
    per_model_imp[name] = ser / ser.sum()          # normalize so scales don't dominate the average
    print(f"\nTop 20 by |SHAP| -- {name.upper()}:")
    for fname, val in per_model_imp[name].head(20).items():
        print(f"  {fname:<26} {val:8.5f}")

# 5) average the three normalized rankings
imp_ser = pd.concat(per_model_imp.values(), axis=1).fillna(0.0).mean(axis=1).sort_values(ascending=False)
print(f"\nTop 20 by averaged (XGB+CatBoost+LightGBM) |SHAP|:")
for fname, val in imp_ser.head(20).items():
    print(f"  {fname:<26} {val:8.5f}")

# 6) cumulative-importance cutoff on the averaged ranking
cum = (imp_ser / imp_ser.sum()).cumsum()
k = max(int((cum < SHAP_CUM).sum()) + 1, MIN_FEATURES)
k = min(k, len(imp_ser))
selected_features = list(imp_ser.index[:k])

print(f"\nRanked {len(imp_ser)} original features; keeping top {k} "
      f"(cumulative avg |SHAP| = {cum.iloc[k-1]:.3f})")
dropped = [c for c in imp_ser.index if c not in selected_features]
print(f"Dropped {len(dropped)}: {dropped}")

# 7) restrict the whole pipeline to the selected features
X      = X[selected_features].reset_index(drop=True)
X_test = X_test[selected_features].reset_index(drop=True)
CAT_COLS = [c for c in CAT_COLS if c in selected_features]
NUM_COLS = [c for c in NUM_COLS if c in selected_features]
print(f"\nFinal feature set: {len(selected_features)}  "
      f"(CAT={len(CAT_COLS)}, NUM={len(NUM_COLS)})")


## 5. Optuna HPO on 10% sample — optimizing balanced accuracy

Runs on the **SHAP-selected** feature set. XGB / CatBoost / LightGBM / LR are tuned here; the
FT-Transformer uses fixed sensible defaults (per-fold HPO on a deep model would be prohibitively slow
inside the outer stack, especially now that the stack also runs per seed).

Sample size is 10% (down from 20%) to offset the added GPU cost of seed-bagging the outer stack and
tuning a fourth model. Each study starts with **15 random trials** before Optuna's TPE sampler begins
optimizing (`n_startup_trials`), and XGBoost/CatBoost/LightGBM all use early stopping during scoring.
Each study's top trials are printed as a small table afterward, to help narrow search ranges by hand
in future runs instead of re-searching blindly.


In [ ]:
X_sample, _, y_sample, _ = train_test_split(X, y, train_size=0.10, stratify=y, random_state=RNG)
X_sample = X_sample.reset_index(drop=True); y_sample = np.asarray(y_sample)


def _lgb_cast_categoricals(Xt, apply_frames, cat_cols):
    """Cast CAT_COLS to 'category' dtype; values unseen in Xt map to an explicit
    'unknown' level (never NaN) -- matches the unknown-slot convention used by
    the target encoder and FT-Transformer elsewhere in this notebook."""
    Xt = Xt.copy()
    outs = [f.copy() for f in apply_frames]
    for c in cat_cols:
        cats = pd.Index(Xt[c].astype(str).unique()).tolist() + ["__unknown__"]
        Xt[c] = pd.Categorical(Xt[c].astype(str), categories=cats)
        for f in outs:
            f[c] = pd.Categorical(f[c].astype(str).where(f[c].astype(str).isin(cats), "__unknown__"),
                                  categories=cats)
    return (Xt, *outs)


def cv_score_xgb(p):
    skf, sc = StratifiedKFold(3, shuffle=True, random_state=RNG), []
    for tr, va in skf.split(X_sample, y_sample):
        Xt, Xv = X_sample.iloc[tr].reset_index(drop=True), X_sample.iloc[va].reset_index(drop=True)
        yt, yv = y_sample[tr], y_sample[va]
        Xt_e, Xv_e = oof_target_encode_multiclass(Xt, yt, Xv, CAT_COLS, N_CLASSES, n_splits=3)
        m = xgb.XGBClassifier(**p, objective="multi:softprob", num_class=N_CLASSES,
                              eval_metric="mlogloss", **xgb_device_kwargs(), random_state=RNG,
                              n_jobs=-1, early_stopping_rounds=50, verbosity=0)
        m.fit(Xt_e, yt, sample_weight=balanced_sample_weight(yt), eval_set=[(Xv_e, yv)], verbose=False)
        sc.append(balanced_accuracy_score(yv, m.predict(Xv_e)))
    return float(np.mean(sc))


def cv_score_cat(p):
    skf, sc = StratifiedKFold(3, shuffle=True, random_state=RNG), []
    for tr, va in skf.split(X_sample, y_sample):
        Xt, Xv = X_sample.iloc[tr].reset_index(drop=True), X_sample.iloc[va].reset_index(drop=True)
        yt, yv = y_sample[tr], y_sample[va]
        m = CatBoostClassifier(**p, loss_function="MultiClass", eval_metric="TotalF1",
                               auto_class_weights="Balanced", random_seed=RNG,
                               **cat_device_kwargs(), verbose=0, allow_writing_files=False)
        m.fit(Xt, yt, cat_features=CAT_COLS, eval_set=(Xv, yv),
              use_best_model=True, early_stopping_rounds=50)
        sc.append(balanced_accuracy_score(yv, m.predict(Xv).ravel().astype(int)))
    return float(np.mean(sc))


def cv_score_lgb(p):
    skf, sc = StratifiedKFold(3, shuffle=True, random_state=RNG), []
    for tr, va in skf.split(X_sample, y_sample):
        Xt, Xv = X_sample.iloc[tr].reset_index(drop=True), X_sample.iloc[va].reset_index(drop=True)
        yt, yv = y_sample[tr], y_sample[va]
        Xt_c, Xv_c = _lgb_cast_categoricals(Xt, [Xv], CAT_COLS)
        m = lgb.LGBMClassifier(**p, objective="multiclass", num_class=N_CLASSES,
                               random_state=RNG, subsample_freq=1,
                               **lgb_device_kwargs(), n_jobs=-1, verbosity=-1)
        m.fit(Xt_c, yt, sample_weight=balanced_sample_weight(yt), eval_set=[(Xv_c, yv)],
              categorical_feature=CAT_COLS, callbacks=[lgb.early_stopping(50, verbose=False)])
        sc.append(balanced_accuracy_score(yv, m.predict(Xv_c)))
    return float(np.mean(sc))


def cv_score_lr(p):
    skf, sc = StratifiedKFold(3, shuffle=True, random_state=RNG), []
    for tr, va in skf.split(X_sample, y_sample):
        Xt, Xv = X_sample.iloc[tr].reset_index(drop=True), X_sample.iloc[va].reset_index(drop=True)
        yt, yv = y_sample[tr], y_sample[va]
        Xt_e, Xv_e = oof_target_encode_multiclass(Xt, yt, Xv, CAT_COLS, N_CLASSES, n_splits=3)
        # LR can't handle NaNs (unlike trees): median-impute then scale, fit on train only.
        imp = SimpleImputer(strategy="median").fit(Xt_e)
        scaler = StandardScaler().fit(imp.transform(Xt_e))
        m = LogisticRegression(**p, class_weight="balanced", max_iter=2000, n_jobs=-1)
        m.fit(scaler.transform(imp.transform(Xt_e)), yt)
        sc.append(balanced_accuracy_score(yv, m.predict(scaler.transform(imp.transform(Xv_e)))))
    return float(np.mean(sc))


def obj_xgb(t):
    return cv_score_xgb({
        "n_estimators":     t.suggest_int("n_estimators", 200, 1000),
        "learning_rate":    t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":        t.suggest_int("max_depth", 3, 10),
        "min_child_weight": t.suggest_float("min_child_weight", 1.0, 20.0),
        "subsample":        t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        t.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "reg_lambda":       t.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
        "gamma":            t.suggest_float("gamma", 1e-8, 5.0, log=True)})

def obj_cat(t):
    return cv_score_cat({
        "iterations":    t.suggest_int("iterations", 300, 1200),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth":         t.suggest_int("depth", 4, 10),
        "l2_leaf_reg":   t.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True)})

def obj_lgb(t):
    return cv_score_lgb({
        "n_estimators":      t.suggest_int("n_estimators", 200, 1000),
        "learning_rate":     t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":         t.suggest_int("max_depth", 3, 10),
        "num_leaves":        t.suggest_int("num_leaves", 15, 127),
        "min_child_samples": t.suggest_int("min_child_samples", 5, 100),
        "subsample":         t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  t.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         t.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "reg_lambda":        t.suggest_float("reg_lambda", 1e-8, 5.0, log=True)})

def obj_lr(t):
    return cv_score_lr({"C": t.suggest_float("C", 1e-3, 10.0, log=True),
                        "solver": "lbfgs", "random_state": RNG})


TRIAL_COLS = ["number", "value", "duration"]
studies = {}
for name, obj, n in [("xgb", obj_xgb, N_TRIALS), ("cat", obj_cat, N_TRIALS),
                      ("lgb", obj_lgb, N_TRIALS), ("lr", obj_lr, 15)]:
    t0 = time.time()
    sampler = optuna.samplers.TPESampler(seed=RNG, n_startup_trials=15)  # random trials before TPE kicks in
    s = optuna.create_study(direction="maximize", sampler=sampler)
    s.optimize(obj, n_trials=n, show_progress_bar=False)
    studies[name] = s
    print(f"  {name}: bal_acc={s.best_value:.5f}  ({time.time()-t0:.1f}s)")
    tdf = s.trials_dataframe()
    param_cols = [c for c in tdf.columns if c.startswith("params_")]
    print(tdf[TRIAL_COLS + param_cols].sort_values("value", ascending=False).head(15).to_string(index=False))
best = {k: v.best_params for k, v in studies.items()}


## 6. Base-learner fit functions (incl. FT-Transformer and LightGBM)

Each returns `(val_proba, test_proba)`, both shape `(n, K)`, with `yv` passed explicitly for early
stopping. Every `fit_*` function now also takes a `seed` argument (default `RNG`) that drives its own
random_state/torch seed — the outer seed-bagging loop below calls each learner once per seed in
`SEEDS`, and threading the seed all the way in (not just through the fold split) means each seed
iteration is a genuinely different model init, not just a different train/val partition.

### FT-Transformer — the neural base learner (from scratch, no external deps)
**FT-Transformer** (Gorishniy et al., 2021) is a Transformer for tabular data. Its key idea is the
**Feature Tokenizer**: *every* feature — categorical **and** numerical — becomes a `d`-dimensional
token. A categorical value is an embedding lookup; a numerical scalar `x_j` becomes `b_j + W_j·x_j`
with its **own** learned `W_j, b_j`. A learned **[CLS]** token is prepended, the token sequence is
run through a stack of pre-LN Transformer encoder blocks, and only the final [CLS] embedding feeds
the classification head. (This is exactly why it tends to beat TabTransformer, which only attends
over categoricals and bolts the numerics on at the end.)

This implementation is ported from a previous contest's binary FT-Transformer and adapted to this
problem:
1. **Multiclass head + softmax.** The head outputs `K=N_CLASSES` logits; training uses
   cross-entropy and inference a softmax, so output column `j` is class `j` — no reordering needed.
2. **Balanced cross-entropy** (`weight = N / (K · count_k)`) — the neural analogue of the trees'
   balanced sample weights, since the target is heavily imbalanced.
3. **Early stopping on validation *balanced* accuracy** (the competition metric), keeping the best
   epoch's weights, with a cosine LR schedule and gradient clipping.
4. **Categoricals → integer index with an "unknown" slot** per column (fit on the training fold;
   categories unseen in that fold map to the reserved index), so no lookup ever goes out of range.
5. **Numerics → median-impute + z-score** on train-fold statistics only. Imputation is required
   because the numerical tokenizer (`b + W·x`) can't consume NaN; the z-score keeps tokens at a sane
   scale (the paper notes the numerical embedding is scale-sensitive and can blow up on raw values).

The model classes (`FeatureTokenizer`, `FTBlock`, `FTTransformerMulti`) are defined once, and
`fit_ft` builds and trains a fresh instance per outer fold. It uses a lower LR (1e-4) than a typical
MLP, following the paper.

### LightGBM — the fifth base learner
Fast, leaf-wise-growth gradient boosting, added mainly for **mechanism diversity**: it's the only
learner using both leaf-wise tree growth *and* native categorical handling with an explicit
`"__unknown__"` sentinel level for values unseen at train time (the same unknown-slot convention used
by the FT-Transformer above and the target encoder). Class imbalance uses the same
`balanced_sample_weight` convention as XGBoost/LR, for consistency with what's actually tuned in HPO.


In [ ]:
def fit_xgb(Xt, yt, Xv, Xte, yv, seed=RNG):
    Xt_e, Xv_e  = oof_target_encode_multiclass(Xt, yt, Xv,  CAT_COLS, N_CLASSES, n_splits=5, seed=seed)
    _,    Xte_e = oof_target_encode_multiclass(Xt, yt, Xte, CAT_COLS, N_CLASSES, n_splits=5, seed=seed)
    m = xgb.XGBClassifier(**best["xgb"], objective="multi:softprob", num_class=N_CLASSES,
                          eval_metric="mlogloss", **xgb_device_kwargs(), random_state=seed,
                          n_jobs=-1, early_stopping_rounds=50, verbosity=0)
    m.fit(Xt_e, yt, sample_weight=balanced_sample_weight(yt), eval_set=[(Xv_e, yv)], verbose=False)
    return m.predict_proba(Xv_e), m.predict_proba(Xte_e)


def fit_cat(Xt, yt, Xv, Xte, yv, seed=RNG):
    m = CatBoostClassifier(**best["cat"], loss_function="MultiClass", eval_metric="TotalF1",
                           auto_class_weights="Balanced", random_seed=seed,
                           **cat_device_kwargs(), verbose=0, allow_writing_files=False)
    m.fit(Xt, yt, cat_features=CAT_COLS, eval_set=(Xv, yv),
          use_best_model=True, early_stopping_rounds=50)
    return m.predict_proba(Xv), m.predict_proba(Xte)


def fit_lr(Xt, yt, Xv, Xte, yv, seed=RNG):
    Xt_e, Xv_e  = oof_target_encode_multiclass(Xt, yt, Xv,  CAT_COLS, N_CLASSES, n_splits=5, seed=seed)
    _,    Xte_e = oof_target_encode_multiclass(Xt, yt, Xte, CAT_COLS, N_CLASSES, n_splits=5, seed=seed)
    imp = SimpleImputer(strategy="median").fit(Xt_e)
    scaler = StandardScaler().fit(imp.transform(Xt_e))
    m = LogisticRegression(**best["lr"], class_weight="balanced", max_iter=2000,
                           n_jobs=-1, random_state=seed)
    m.fit(scaler.transform(imp.transform(Xt_e)), yt)
    return (m.predict_proba(scaler.transform(imp.transform(Xv_e))),
            m.predict_proba(scaler.transform(imp.transform(Xte_e))))


def fit_lgb(Xt, yt, Xv, Xte, yv, seed=RNG):
    Xt_c, Xv_c, Xte_c = _lgb_cast_categoricals(Xt, [Xv, Xte], CAT_COLS)
    m = lgb.LGBMClassifier(**best["lgb"], objective="multiclass", num_class=N_CLASSES,
                           random_state=seed, subsample_freq=1,
                           **lgb_device_kwargs(), n_jobs=-1, verbosity=-1)
    m.fit(Xt_c, yt, sample_weight=balanced_sample_weight(yt), eval_set=[(Xv_c, yv)],
          categorical_feature=CAT_COLS, callbacks=[lgb.early_stopping(50, verbose=False)])
    return m.predict_proba(Xv_c), m.predict_proba(Xte_c)


# ===========================================================================
# FT-Transformer (Gorishniy et al., 2021) — from-scratch PyTorch, multiclass.
# No third-party tabular-DL packages; only torch (preinstalled on Kaggle).
# ===========================================================================
FT_D_TOKEN, FT_N_HEADS, FT_N_LAYERS = 64, 8, 3
FT_FFN_MULT = 4 / 3
FT_ATTN_DROP, FT_FFN_DROP, FT_RES_DROP, FT_HEAD_DROP = 0.2, 0.1, 0.0, 0.0
FT_BATCH, FT_EPOCHS, FT_LR, FT_WD, FT_PATIENCE = 1024, 40, 1e-4, 1e-5, 6

class FeatureTokenizer(nn.Module):
    """Turns every feature (numeric AND categorical) into a d-dim token."""
    def __init__(self, cat_cardinalities, n_num, d_token):
        super().__init__()
        self.n_num = n_num
        if n_num > 0:
            self.num_weight = nn.Parameter(torch.empty(n_num, d_token))
            self.num_bias   = nn.Parameter(torch.empty(n_num, d_token))
            nn.init.kaiming_uniform_(self.num_weight, a=5 ** 0.5)
            nn.init.kaiming_uniform_(self.num_bias,   a=5 ** 0.5)
        self.cat_embeds = nn.ModuleList([nn.Embedding(c, d_token) for c in cat_cardinalities])
        if len(cat_cardinalities) > 0:
            self.cat_bias = nn.Parameter(torch.empty(len(cat_cardinalities), d_token))
            for e in self.cat_embeds:
                nn.init.kaiming_uniform_(e.weight, a=5 ** 0.5)
            nn.init.kaiming_uniform_(self.cat_bias, a=5 ** 0.5)

    def forward(self, x_cat, x_num):
        toks = []
        if self.n_num > 0 and x_num.shape[1] > 0:
            # (B, n_num, 1) * (1, n_num, d) + (1, n_num, d) -> (B, n_num, d)
            toks.append(x_num.unsqueeze(-1) * self.num_weight.unsqueeze(0) + self.num_bias.unsqueeze(0))
        if len(self.cat_embeds) > 0:
            parts = [self.cat_embeds[j](x_cat[:, j]) for j in range(len(self.cat_embeds))]
            toks.append(torch.stack(parts, dim=1) + self.cat_bias.unsqueeze(0))
        return torch.cat(toks, dim=1)

class FTBlock(nn.Module):
    """Pre-LN encoder block; the paper drops the first LN of the first block."""
    def __init__(self, d, n_heads, ffn_mult, attn_drop, ffn_drop, res_drop, first_block):
        super().__init__()
        self.first_block = first_block
        if not first_block:
            self.ln1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=attn_drop, batch_first=True)
        self.drop1 = nn.Dropout(res_drop)
        self.ln2 = nn.LayerNorm(d)
        hidden = int(d * ffn_mult)
        self.ffn = nn.Sequential(
            nn.Linear(d, hidden), nn.GELU(), nn.Dropout(ffn_drop),
            nn.Linear(hidden, d), nn.Dropout(res_drop))

    def forward(self, x):
        h = x if self.first_block else self.ln1(x)
        a, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop1(a)
        x = x + self.ffn(self.ln2(x))
        return x

class FTTransformerMulti(nn.Module):
    """Feature tokenizer + [CLS] + transformer stack + K-way softmax head."""
    def __init__(self, cat_cardinalities, n_num, n_classes,
                 d_token=FT_D_TOKEN, n_heads=FT_N_HEADS, n_layers=FT_N_LAYERS,
                 ffn_mult=FT_FFN_MULT, attn_drop=FT_ATTN_DROP, ffn_drop=FT_FFN_DROP,
                 res_drop=FT_RES_DROP, head_drop=FT_HEAD_DROP):
        super().__init__()
        self.tokenizer = FeatureTokenizer(cat_cardinalities, n_num, d_token)
        self.cls = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)
        self.blocks = nn.ModuleList([
            FTBlock(d_token, n_heads, ffn_mult, attn_drop, ffn_drop, res_drop,
                    first_block=(i == 0)) for i in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_token)
        self.head = nn.Sequential(nn.ReLU(), nn.Dropout(head_drop),
                                  nn.Linear(d_token, n_classes))

    def forward(self, x_cat, x_num):
        tokens = self.tokenizer(x_cat, x_num)               # (B, k, d)
        cls = self.cls.expand(tokens.size(0), -1, -1)       # (B, 1, d)
        x = torch.cat([cls, tokens], dim=1)                 # (B, 1+k, d)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.final_norm(x[:, 0]))          # (B, n_classes)

def fit_ft(Xt, yt, Xv, Xte, yv, seed=RNG):
    torch.manual_seed(seed); np.random.seed(seed)
    dev = torch.device(TORCH_DEVICE)
    cat_here = [c for c in CAT_COLS if c in Xt.columns]
    num_here = [c for c in NUM_COLS if c in Xt.columns]

    def _s(series):                          # robust NaN->str (works on pandas 2 & 3)
        return series.astype(str).fillna("__nan__")

    # --- categoricals -> integer index (train vocab; unseen -> reserved slot) ---
    cat_maps, cat_card = {}, []
    for col in cat_here:
        vals = sorted(_s(Xt[col]).unique())
        cat_maps[col] = {v: i for i, v in enumerate(vals)}
        cat_card.append(len(vals) + 1)                       # +1 = unknown slot
    def enc_cat(dfx):
        if not cat_here:
            return np.zeros((len(dfx), 0), dtype=np.int64)
        a = np.zeros((len(dfx), len(cat_here)), dtype=np.int64)
        for j, col in enumerate(cat_here):
            unk = len(cat_maps[col])
            a[:, j] = _s(dfx[col]).map(cat_maps[col]).fillna(unk).astype(np.int64).values
        return a

    # --- numerics -> train-median impute + train-stats z-score ---
    if num_here:
        med = Xt[num_here].median()
        mu  = Xt[num_here].fillna(med).mean().values.astype(np.float32)
        sd  = Xt[num_here].fillna(med).std().replace(0, 1).values.astype(np.float32)
        def enc_num(dfx):
            return (dfx[num_here].fillna(med).values.astype(np.float32) - mu) / sd
    else:
        def enc_num(dfx):
            return np.zeros((len(dfx), 0), dtype=np.float32)

    def to_dev(c, n):
        return (torch.as_tensor(c, dtype=torch.long,    device=dev),
                torch.as_tensor(n, dtype=torch.float32, device=dev))
    Xc_tr, Xn_tr = to_dev(enc_cat(Xt),  enc_num(Xt))
    Xc_va, Xn_va = to_dev(enc_cat(Xv),  enc_num(Xv))
    Xc_te, Xn_te = to_dev(enc_cat(Xte), enc_num(Xte))
    y_tr = torch.as_tensor(yt, dtype=torch.long, device=dev)

    model = FTTransformerMulti(cat_card, len(num_here), N_CLASSES).to(dev)

    # balanced cross-entropy (neural analogue of balanced sample weights)
    cnt = np.bincount(yt, minlength=N_CLASSES).astype(float)
    cw  = torch.as_tensor(len(yt) / (N_CLASSES * np.clip(cnt, 1, None)),
                          dtype=torch.float32, device=dev)
    crit  = nn.CrossEntropyLoss(weight=cw)
    opt   = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=FT_WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)

    def proba(Xc, Xn, bs=4096):
        model.eval(); outs = []
        with torch.no_grad():
            for s in range(0, len(Xc), bs):
                outs.append(model(Xc[s:s + bs], Xn[s:s + bs]).softmax(1).cpu().numpy())
        return np.concatenate(outs) if outs else np.zeros((0, N_CLASSES), dtype=np.float32)

    n = len(yt); idx = np.arange(n)
    best_ba, best_state, bad = -1.0, None, 0
    for _ in range(FT_EPOCHS):
        model.train(); np.random.shuffle(idx)
        for s in range(0, n, FT_BATCH):
            b = torch.as_tensor(idx[s:s + FT_BATCH], dtype=torch.long, device=dev)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(Xc_tr[b], Xn_tr[b]), y_tr[b])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        ba = balanced_accuracy_score(yv, proba(Xc_va, Xn_va).argmax(1))
        if ba > best_ba:
            best_ba, bad = ba, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= FT_PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return proba(Xc_va, Xn_va), proba(Xc_te, Xn_te)


## 7. Seed-bagged outer 5-fold stacking (5 base learners)

Meta feature matrix is `5 × K` wide; all predictions are out-of-fold. The entire 5-fold stacking pass
(`run_stack_for_seed`) runs once per seed in `SEEDS`, with every learner's own randomness (not just the
fold split) varying by seed; `oof_meta`/`test_meta` are then averaged across seeds before calibration
and the meta-learner, trading `len(SEEDS)`× the GPU time for lower OOF/test variance. **The
FT-Transformer is still the slow branch** — on CPU with full data, budget accordingly; on GPU it's
much faster, but this is now the dominant cost driver given the seed multiplier.


In [ ]:
BASE_LEARNERS = ["xgb", "cat", "lr", "ft", "lgb"]
N_BASE = len(BASE_LEARNERS)
FIT_FN = {"xgb": fit_xgb, "cat": fit_cat, "lr": fit_lr, "ft": fit_ft, "lgb": fit_lgb}
def _slot(i): return slice(i * N_CLASSES, (i + 1) * N_CLASSES)

def run_stack_for_seed(seed):
    """One full outer 5-fold stacking pass. Returns (oof_meta, test_meta, fold_scores)."""
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed)
    oof_meta_s  = np.zeros((len(X), N_BASE * N_CLASSES))
    test_meta_s = np.zeros((len(X_test), N_BASE * N_CLASSES))
    fold_scores_s = {k: [] for k in BASE_LEARNERS}

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        Xt = X.iloc[tr_idx].reset_index(drop=True); Xv = X.iloc[va_idx].reset_index(drop=True)
        yt, yv = y[tr_idx], y[va_idx]; t0 = time.time()

        preds = [FIT_FN[k](Xt, yt, Xv, X_test, yv, seed=seed) for k in BASE_LEARNERS]

        for i, (pv, pt) in enumerate(preds):
            oof_meta_s[va_idx, _slot(i)] = pv
            test_meta_s[:, _slot(i)]    += pt / N_FOLDS

        for i, k in enumerate(BASE_LEARNERS):
            fold_scores_s[k].append(balanced_accuracy_score(yv, oof_meta_s[va_idx, _slot(i)].argmax(1)))
        print(f"    fold {fold} ({time.time()-t0:.1f}s)  "
              + "  ".join(f"{k.upper()}={fold_scores_s[k][-1]:.5f}" for k in BASE_LEARNERS))
    return oof_meta_s, test_meta_s, fold_scores_s


oof_meta  = np.zeros((len(X), N_BASE * N_CLASSES))
test_meta = np.zeros((len(X_test), N_BASE * N_CLASSES))
seed_base_oof = {k: [] for k in BASE_LEARNERS}     # per-seed OOF balanced accuracy, for the run log

for seed in SEEDS:
    print(f"  Seed {seed}:")
    oof_s, test_s, fold_scores_s = run_stack_for_seed(seed)
    oof_meta  += oof_s  / len(SEEDS)
    test_meta += test_s / len(SEEDS)
    for i, k in enumerate(BASE_LEARNERS):
        seed_ba = balanced_accuracy_score(y, oof_s[:, _slot(i)].argmax(1))
        seed_base_oof[k].append(seed_ba)
        print(f"    {k:>7} OOF (seed {seed}): {seed_ba:.5f}")

print(f"\nBase-learner OOF balanced accuracy (averaged over {len(SEEDS)} seeds):")
base_oof_scores = {}
for i, k in enumerate(BASE_LEARNERS):
    ba = balanced_accuracy_score(y, oof_meta[:, _slot(i)].argmax(1))
    base_oof_scores[k] = ba
    print(f"  {k:>7}: {ba:.5f}  (per-seed: {[round(s, 5) for s in seed_base_oof[k]]})")


## 7.5 Cross-fitted calibration of base-learner OOF probabilities

`oof_meta`/`test_meta` hold each base learner's probabilities pre-calibration. Class-balancing
(sample weights, CatBoost's `auto_class_weights`) systematically distorts predicted probabilities in
exchange for better decision boundaries — stacking on distorted probabilities can make the
meta-learner's own `class_weight="balanced"` double-correct. We cross-fit an isotonic calibrator
(one-vs-rest per class, renormalized) per base learner: every OOF point is calibrated by a model that
never saw its label, so this stays leakage-safe by the same logic as the meta-learner's own OOF loop
below. Kept only if it improves meta-learner OOF balanced accuracy (validated, not assumed).


In [ ]:
def _calibrate_block(oof_block, y_true, test_block, n_splits=5, seed=RNG):
    """Cross-fit one-vs-rest isotonic calibration on a single base learner's K
    OOF probability columns. Returns (calibrated_oof, calibrated_test)."""
    n_classes = oof_block.shape[1]
    cal_oof = np.zeros_like(oof_block)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, va in kf.split(oof_block):
        for k in range(n_classes):
            ir = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
            ir.fit(oof_block[tr, k], (y_true[tr] == k).astype(float))
            cal_oof[va, k] = ir.predict(oof_block[va, k])
    row_sum = cal_oof.sum(axis=1, keepdims=True)
    cal_oof = np.divide(cal_oof, row_sum, out=np.full_like(cal_oof, 1.0 / n_classes), where=row_sum > 0)

    cal_test = np.zeros_like(test_block)
    for k in range(n_classes):
        ir = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
        ir.fit(oof_block[:, k], (y_true == k).astype(float))
        cal_test[:, k] = ir.predict(test_block[:, k])
    row_sum = cal_test.sum(axis=1, keepdims=True)
    cal_test = np.divide(cal_test, row_sum, out=np.full_like(cal_test, 1.0 / n_classes), where=row_sum > 0)
    return cal_oof, cal_test


oof_meta_cal  = oof_meta.copy()
test_meta_cal = test_meta.copy()
for i, k in enumerate(BASE_LEARNERS):
    cal_oof, cal_test = _calibrate_block(oof_meta[:, _slot(i)], y, test_meta[:, _slot(i)])
    oof_meta_cal[:, _slot(i)]  = cal_oof
    test_meta_cal[:, _slot(i)] = cal_test
    print(f"  {k:>7} OOF bal_acc  raw={balanced_accuracy_score(y, oof_meta[:, _slot(i)].argmax(1)):.5f}"
          f"  calibrated={balanced_accuracy_score(y, oof_meta_cal[:, _slot(i)].argmax(1)):.5f}")


def _meta_oof_bal_acc(block):
    """Quick OOF balanced accuracy of the meta-learner trained on a given oof_meta variant."""
    proba = np.zeros((len(y), N_CLASSES))
    for tr, va in StratifiedKFold(N_FOLDS, shuffle=True, random_state=RNG + 1).split(block, y):
        m = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, n_jobs=-1, random_state=RNG)
        m.fit(block[tr], y[tr])
        proba[va] = m.predict_proba(block[va])
    return balanced_accuracy_score(y, proba.argmax(1))

ba_raw = _meta_oof_bal_acc(oof_meta)
ba_cal = _meta_oof_bal_acc(oof_meta_cal)
CALIBRATE_BASE = ba_cal > ba_raw
print(f"\n  Meta OOF bal_acc  raw={ba_raw:.5f}  calibrated={ba_cal:.5f}  -> using calibration: {CALIBRATE_BASE}")

if CALIBRATE_BASE:
    oof_meta, test_meta = oof_meta_cal, test_meta_cal


## 8. Meta-learner + per-class decision-weight search

Trains on `oof_meta` (calibrated or not, per the toggle above). The meta-contribution readout (sum of
mean |coef| over each learner's class columns) tells you whether a given base learner is earning its
slot — a near-zero contribution means the meta-learner found it redundant with the trees, which is
fine, the stack ignores it gracefully.

The old binary prior-correction toggle (`meta_test_proba / priors`, used only if it beat the
uncorrected OOF score) is replaced by a general **coordinate-ascent grid search** over a per-class
decision-weight vector `w`: for a few rounds, each class's multiplier is scanned over a grid holding
the others fixed, keeping whichever value most improves OOF balanced accuracy. The result is compared
against `w = ones` (no correction) and `w = 1/priors` (the old heuristic), and whichever wins on OOF is
used for the final submission — this strictly generalizes the old toggle rather than replacing its
logic outright.


In [ ]:
# --- meta-learner: keep class_weight="balanced" (v5 showed dropping it collapses balanced
#     accuracy toward the majority class); sweep C. This is one blend candidate among several. ---
skf_meta = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RNG + 1)
def _fit_meta(C):
    """Full skf_meta OOF pass for one C. Returns (oof_proba, test_proba, mean|coef|)."""
    oof = np.zeros((len(X), N_CLASSES)); test = np.zeros((len(X_test), N_CLASSES)); coefs = []
    for tr, va in skf_meta.split(oof_meta, y):
        m = LogisticRegression(C=C, class_weight="balanced", max_iter=2000, n_jobs=-1, random_state=RNG)
        m.fit(oof_meta[tr], y[tr])
        oof[va] = m.predict_proba(oof_meta[va])
        test   += m.predict_proba(test_meta) / N_FOLDS
        coefs.append(np.abs(m.coef_).mean(axis=0))
    return oof, test, np.mean(coefs, axis=0)

meta_C_grid = [0.1, 0.3, 1.0, 3.0, 10.0]
meta_by_C = {}
for C in meta_C_grid:
    oof_c, test_c, mc_c = _fit_meta(C)
    ba_c = balanced_accuracy_score(y, oof_c.argmax(1))
    meta_by_C[C] = (ba_c, oof_c, test_c, mc_c)
    print(f"  meta C={C:<5}: OOF bal_acc={ba_c:.5f}")
meta_C = max(meta_by_C, key=lambda c: meta_by_C[c][0])
meta_bal_acc, meta_oof_proba, meta_test_proba, mc = meta_by_C[meta_C]
print(f"  -> best meta C={meta_C}  (OOF bal_acc={meta_bal_acc:.5f})")
print("  Meta contribution by base learner:")
for i, k in enumerate(BASE_LEARNERS):
    print(f"    {k:>7}: {mc[_slot(i)].sum():.3f}")

# --- blend candidates (v5 showed the 5-way simple average is dragged down by the weak LR;
#     compare a few clean blends and pick the best OOF -- few DOF, so low overfit risk) ---
def _avg(idxs, mat):
    return np.mean([mat[:, _slot(i)] for i in idxs], axis=0)

best_base_oof = max(base_oof_scores.values())
STRONG = [i for i, k in enumerate(BASE_LEARNERS) if base_oof_scores[k] >= best_base_oof - 0.01]
best_i = int(np.argmax([base_oof_scores[k] for k in BASE_LEARNERS]))
print(f"  strong learners (within 0.01 of best): {[BASE_LEARNERS[i] for i in STRONG]}"
      f"   best single: {BASE_LEARNERS[best_i]}")

avg_oof  = _avg(range(N_BASE), oof_meta)          # all 5 (kept as the simple_avg reference)
avg_test = _avg(range(N_BASE), test_meta)
avg_bal_acc = balanced_accuracy_score(y, avg_oof.argmax(1))

blend_candidates = {
    "avg_all":    (avg_oof, avg_test),
    "avg_strong": (_avg(STRONG, oof_meta), _avg(STRONG, test_meta)),
    "meta":       (meta_oof_proba, meta_test_proba),
    "best_base":  (oof_meta[:, _slot(best_i)], test_meta[:, _slot(best_i)]),
}
blend_scores = {name: balanced_accuracy_score(y, o.argmax(1)) for name, (o, t) in blend_candidates.items()}
blend_source = max(blend_scores, key=blend_scores.get)
blend_oof, blend_test = blend_candidates[blend_source]
blend_bal_acc = blend_scores[blend_source]
print("  Blend candidates (OOF): " + "  ".join(f"{n}={v:.5f}" for n, v in blend_scores.items()))
print(f"  -> blend: {blend_source}  ({blend_bal_acc:.5f})")

# --- per-class decision-weight search on the chosen blend, guarded against OOF overfit ---
priors = np.array([(y == k).mean() for k in range(N_CLASSES)])
WEIGHT_MARGIN = 5e-4   # only adopt reweighting if it beats 'none' by more than this on OOF

def optimize_class_weights(y_true, proba, n_rounds=3, grid=np.linspace(0.75, 1.35, 25)):
    w = np.ones(proba.shape[1])
    best_score = balanced_accuracy_score(y_true, proba.argmax(1))
    for _ in range(n_rounds):
        improved = False
        for k in range(proba.shape[1]):
            best_wk, best_local = w[k], best_score
            for cand in grid:
                w_try = w.copy(); w_try[k] = cand
                s = balanced_accuracy_score(y_true, (proba * w_try).argmax(1))
                if s > best_local:
                    best_local, best_wk = s, cand
            if best_local > best_score:
                w[k] = best_wk; best_score = best_local; improved = True
        if not improved:
            break
    return w, best_score

w_search, ba_search = optimize_class_weights(y, blend_oof)
w_prior = 1.0 / priors
ba_prior = balanced_accuracy_score(y, (blend_oof * w_prior).argmax(1))
ba_none  = blend_bal_acc

# 'none' is always eligible; reweighting must beat it by WEIGHT_MARGIN to be adopted
eligible = {"none": (np.ones(N_CLASSES), ba_none)}
if ba_prior  >= ba_none + WEIGHT_MARGIN: eligible["prior_inverse"] = (w_prior,  ba_prior)
if ba_search >= ba_none + WEIGHT_MARGIN: eligible["search"]        = (w_search, ba_search)
class_weight_method = max(eligible, key=lambda k: eligible[k][1])
class_weights, best_adj_bal_acc = eligible[class_weight_method]
print("  Class-weight candidates (OOF): "
      + f"none={ba_none:.5f}  prior_inverse={ba_prior:.5f}  search={ba_search:.5f}"
      + f"  (margin={WEIGHT_MARGIN})")
print(f"  -> using: {class_weight_method}  (bal_acc={best_adj_bal_acc:.5f}, "
      f"weights={np.round(class_weights, 3).tolist()})")

print("\n=== Summary ===")
for i, k in enumerate(BASE_LEARNERS):
    print(f"  {k:>7} OOF   : {balanced_accuracy_score(y, oof_meta[:, _slot(i)].argmax(1)):.5f}")
print(f"  simple avg  : {avg_bal_acc:.5f}")
print(f"  stacked meta: {meta_bal_acc:.5f}  (C={meta_C})")
print(f"  blend ({blend_source}) + class-weight ({class_weight_method}): {best_adj_bal_acc:.5f}")

## 9. Submission

In [ ]:
final_pred = (blend_test * class_weights).argmax(1)
sub = pd.DataFrame({"id": test_ids, TARGET: [int_to_cls[i] for i in final_pred]})
sub.to_csv("submission.csv", index=False)
print("Submission shape:", sub.shape); print(sub.head())
print("\nPredicted class balance:")
print(sub[TARGET].value_counts(normalize=True))

run_metrics = {
    "n_seeds": len(SEEDS),
    "seeds": SEEDS,
    "hpo_sample_frac": 0.10,
    "notebook_runtime_sec": round(time.time() - _NOTEBOOK_T0, 1),
    "n_features_selected": len(selected_features),
    "base_oof_bal_acc": {k: round(float(base_oof_scores[k]), 5) for k in BASE_LEARNERS},
    "meta_oof_bal_acc_raw": round(float(meta_bal_acc), 5),
    "calibration_used": bool(CALIBRATE_BASE),
    "meta_oof_bal_acc_calibrated": round(float(ba_cal), 5),
    "meta_C": meta_C,
    "blend_source": blend_source,
    "simple_avg_oof_bal_acc": round(float(avg_bal_acc), 5),
    "class_weight_method": class_weight_method,
    "class_weights": np.round(class_weights, 4).tolist(),
    "final_oof_bal_acc": round(float(best_adj_bal_acc), 5),
    "predicted_class_balance": sub[TARGET].value_counts(normalize=True).round(4).to_dict(),
}
print("\nRUN_METRICS_JSON:" + json.dumps(run_metrics))
